# 07 — AMAI Household Component Imputation Using ENIGH

This notebook estimates the AMAI household components that cannot be obtained directly from the Origin-Destination (OD) survey.

The previous notebook constructed the head-of-household education component and added the AMAI scores associated with education, Internet access, and cars or trucks. Three additional AMAI components are required:

1. number of household members aged 14 or older who worked during the previous month;
2. number of complete bathrooms; and
3. number of rooms used for sleeping.

These variables are available in the 2022 ENIGH but not directly in the OD survey. We therefore use ENIGH to construct the corresponding AMAI categories and train probabilistic classification models that can be transferred to the OD data.

The notebook follows five main steps:

1. Construct the three target variables from ENIGH.
2. Harmonize predictors available in both ENIGH and the OD survey.
3. Train and evaluate probabilistic models using ENIGH survey weights and grouped cross-validation.
4. Apply the selected models to each OD dwelling.
5. Convert the predicted probability distributions into expected AMAI scores.

Because household income is missing for a substantial share of OD dwellings, two versions of each model are estimated: one that includes income and another that does not.

The main outputs are:

- `od_housing_amai_imputed.csv`: dwelling-level OD dataset containing the predicted category probabilities and expected AMAI scores for the three modeled components.
- `amai_imputation_models.joblib`: fitted models and the information required to reproduce their predictions.

The resulting dataset contains the household-level components required for the subsequent AMAI socioeconomic-level calculation.

In [1]:
import numpy as np
import pandas as pd
import mxcensus

In [2]:
enigh_household_questionnaire = mxcensus.load_enigh(table="hogares", period="2022")
enigh_households = mxcensus.load_enigh_hogares(period="2022")
enigh_population = mxcensus.load_enigh_personas(period="2022")
enigh_housing = mxcensus.load_enigh_viviendas(period="2022")
od_housing = pd.read_csv("/Users/sebastiangutierrezbernal/Desktop/Tec/Ciudades para el futuro/Proyecto Transporte GDL/informal_jobs/outputs/files/housing_data.csv")
od_population = pd.read_csv("/Users/sebastiangutierrezbernal/Desktop/Tec/Ciudades para el futuro/Proyecto Transporte GDL/informal_jobs/outputs/files/head_household_education_level_data.csv")

## 1. Construction of AMAI target variables in ENIGH

### 1.1 Household members aged 14 or older who worked during the previous month

The first component corresponds to the number of household members aged 14 or older who worked during the previous month.

For person $i$ in household $h$, define

$$W_{ih} = \mathbf{1} \left(\mathrm{age}_{ih}\geq14 \;\land\; \mathrm{worked}_{ih}=1 \right).$$

The number of qualifying household members is then

$$N_h^{\mathrm{workers}} = \sum_{i\in h} W_{ih}.$$

This count is converted to the AMAI categories

$$Y_h^{(\mathrm{workers})} \in \{0,1,2,3,4+\}.$$

The variable is constructed from ENIGH because the equivalent information is not directly available at the dwelling level in the OD survey.

In [ ]:
# Identify household members aged 14 or older who worked during the previous month
enigh_population["14_anos_trabajo_mes"] = ((enigh_population["edad"] >= 14) & (enigh_population["trabajo_mp"].str.strip() == "1")).astype("Int64")

# Aggregate the individual indicator to obtain the number of qualifying members per household
workers_per_household = enigh_population.reset_index().groupby(["folioviv", "foliohog"], as_index=False)["14_anos_trabajo_mes"].sum()

# Attach the household-level worker count to the ENIGH household table
enigh_households = enigh_households.reset_index().merge(workers_per_household, on=["folioviv", "foliohog"], how="left", validate="one_to_one")

# Collapse the original count into the categories used by the AMAI scoring system
workers_mapping = {
    0: "0",
    1: "1",
    2: "2",
    3: "3",
    4: "4 o más",
    5: "4 o más",
    6: "4 o más",
    7: "4 o más",
    8: "4 o más",
    9: "4 o más",
    10: "4 o más",
    11: "4 o más"
}

enigh_households["amai_14_anos_trabajo_mes"] = enigh_households["14_anos_trabajo_mes"].map(workers_mapping).astype("string")

### 1.2 Number of complete bathrooms

The second modeled component is the number of complete bathrooms available in the dwelling.

The original ENIGH count is grouped according to the AMAI categories

$$Y_h^{(\mathrm{bathrooms})} = \begin{cases} 0, & B_h=0,\\ 1, & B_h=1,\\ 2+, & B_h\geq2, \end{cases}$$

where $B_h$ denotes the number of complete bathrooms reported for dwelling $h$.

Because the bathroom count is recorded at the dwelling level in ENIGH, it is inherited by every household associated with the same `folioviv`.

In [4]:
# We obtain the number of complete bathrooms directly from the housing database
complete_bathrooms = enigh_housing.reset_index()[["folioviv", "bano_comp"]].copy()
complete_bathrooms["bano_comp"] = pd.to_numeric(complete_bathrooms["bano_comp"], errors="coerce").astype("Int64")

# We inherit the number of complete bathrooms to every household in the dwelling
enigh_households = enigh_households.merge(complete_bathrooms, on="folioviv", how="left", validate="many_to_one")
enigh_households = enigh_households.rename(columns={"bano_comp": "banos_completos"})

In [ ]:
# Convert the original bathroom count to the AMAI categories 0, 1, and 2+
bathrooms_mapping = {
    pd.NA: pd.NA,
    0: "0",
    1: "1",
    2: "2 o más",
    3: "2 o más",
    4: "2 o más",
    5: "2 o más",
    6: "2 o más",
    7: "2 o más",
    8: "2 o más",
    9: "2 o más"
}

enigh_households["amai_banos_completos"] = enigh_households["banos_completos"].map(bathrooms_mapping).astype("string")

Not all ENIGH observations contain a valid bathroom count. These observations cannot be used as labeled examples for the bathroom model and are therefore excluded from its training and evaluation sample.

The following check quantifies the number of available and missing target observations.

In [ ]:
# Only observations with a known bathroom category can be used as labeled model data
known_bathrooms = enigh_households.loc[enigh_households["amai_banos_completos"].notna()].copy()
unknown_bathrooms = enigh_households.loc[enigh_households["amai_banos_completos"].isna()].copy()

print(f"Known bathrooms: {len(known_bathrooms):,}")
print(f"Missing bathrooms: {len(unknown_bathrooms):,}")
print(f"Missing share: {len(unknown_bathrooms) / len(enigh_households) * 100:.2f}%")

Known bathrooms: 88,364
Missing bathrooms: 1,738
Missing share: 1.93%


### 1.3 Number of rooms used for sleeping

The third modeled component is the number of rooms used for sleeping.

Let $R_h$ denote the number of sleeping rooms reported for dwelling $h$. The corresponding AMAI category is

$$Y_h^{(\mathrm{rooms})} = \begin{cases} 1, & R_h=1,\\ 2, & R_h=2,\\ 3, & R_h=3,\\ 4+, & R_h\geq4. \end{cases}$$

As with the bathroom variable, the original ENIGH information is defined at the dwelling level and is inherited by every household within the dwelling.

In [ ]:
# Obtain the number of rooms used for sleeping from the ENIGH dwelling table
sleeping_rooms = enigh_housing.reset_index()[["folioviv", "cuart_dorm"]].copy()

# Attach the dwelling-level sleeping-room count to every household in the dwelling
enigh_households = enigh_households.merge(sleeping_rooms, on="folioviv", how="left", validate="many_to_one")
enigh_households = enigh_households.rename(columns={"cuart_dorm": "cuartos_dormir"})

In [ ]:
# Convert the original sleeping-room count to the AMAI categories 1, 2, 3, and 4+
rooms_mapping = {
    "1": "1",
    "2": "2",
    "3": "3",
    "4": "4 o más",
    "5": "4 o más",
    "6": "4 o más",
    "7": "4 o más",
    "8": "4 o más",
    "9": "4 o más",
    "10": "4 o más"
}

enigh_households["amai_cuartos_dormir"] = enigh_households["cuartos_dormir"].map(rooms_mapping).astype("string")

## 2. Predictor selection and ENIGH–OD harmonization

The models are trained using ENIGH but applied to the OD survey. Consequently, predictors must satisfy two requirements:

1. they must contain information about the target variable; and
2. they must have a comparable representation in both datasets.

We first enrich the ENIGH household data with additional variables from the household questionnaire and dwelling tables. We then examine their association with each of the three target variables before defining the final feature sets.

### 2.1 ENIGH predictor enrichment

The ENIGH household table is supplemented with Internet access and vehicle-availability variables from the household questionnaire, together with housing tenure from the dwelling table.

These additional variables provide household characteristics that can later be harmonized with equivalent variables available in the OD survey.

In [ ]:
# Add household predictors from the ENIGH household questionnaire
enigh_household_predictors = enigh_household_questionnaire.reset_index()[["folioviv", "foliohog", "conex_inte", "num_auto", "num_moto", "num_bici"]].copy()
enigh_households = enigh_households.merge(enigh_household_predictors, on=["folioviv", "foliohog"], how="left", validate="one_to_one")

# Add dwelling-level housing tenure from the ENIGH dwelling table
enigh_housing_predictors = enigh_housing.reset_index()[["folioviv", "tenencia"]].copy()
enigh_households = enigh_households.merge(enigh_housing_predictors, on="folioviv", how="left", validate="many_to_one")

### 2.2 Predictor association screening

Before defining the model feature sets, we examine the association between the available ENIGH variables and each target.

For numerical variables, we use Spearman's rank correlation. For a predictor $X_j$ and a numerical representation of target $Y^{(t)}$,

$$\rho_s(X_j,Y^{(t)}) = \text{corr} \left(\text{rank}(X_j), \text{rank}(Y^{(t)}) \right).$$

For categorical variables, we use the bias-corrected Cramér's $V$, which measures the association between two categorical variables.

In [ ]:
# Numerical association screening using Spearman rank correlation on the original counts
numeric_targets = {
    "Workers": "14_anos_trabajo_mes",
    "Bathrooms": "banos_completos",
    "Rooms": "cuartos_dormir"
}

numeric_features = [
    "edad_jefe",
    "tot_integ",
    "ing_cor",
    "num_auto",
    "num_moto",
    "num_bici"
]

numerical_correlations = {}

for target_name, target_column in numeric_targets.items():
    target_data = enigh_households.loc[enigh_households[target_column].notna()].copy()
    numerical_correlations[target_name] = target_data[numeric_features + [target_column]].corr(method="spearman")[target_column].drop(target_column).sort_values(key=abs, ascending=False)

# Categorical association screening using bias-corrected Cramér's V
from scipy.stats import chi2_contingency
def cramers_v(x, y):
    table = pd.crosstab(x, y)

    if table.empty:
        return np.nan

    chi2 = chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    phi2 = chi2 / n
    r, k = table.shape

    phi2_corrected = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corrected = r - ((r - 1) ** 2) / (n - 1)
    k_corrected = k - ((k - 1) ** 2) / (n - 1)

    denominator = min(k_corrected - 1, r_corrected - 1)

    return np.sqrt(phi2_corrected / denominator) if denominator > 0 else np.nan

categorical_features = [
    "tam_loc",
    "sexo_jefe",
    "educa_jefe",
    "conex_inte",
    "tenencia"
]

categorical_targets = {
    "Workers": "amai_14_anos_trabajo_mes",
    "Bathrooms": "amai_banos_completos",
    "Rooms": "amai_cuartos_dormir"
}
categorical_correlations = {}
for target_name, target_column in categorical_targets.items():
    target_data = enigh_households.loc[enigh_households[target_column].notna()].copy()
    categorical_correlations[target_name] = pd.Series({
        column: cramers_v(target_data[column], target_data[target_column])
        for column in categorical_features
    }).sort_values(ascending=False)

In [11]:
for target_name in numeric_targets:
    print(f"\n{'=' * 60}")
    print(target_name)
    print("=" * 60)

    print("\nNumerical predictors - Spearman")
    print(numerical_correlations[target_name])

    print("\nCategorical predictors - Cramer's V")
    print(categorical_correlations[target_name])


Workers

Numerical predictors - Spearman
tot_integ    0.562682
ing_cor      0.370769
num_moto     0.156341
edad_jefe   -0.126012
num_bici     0.117206
num_auto     0.092463
Name: 14_anos_trabajo_mes, dtype: float64

Categorical predictors - Cramer's V
sexo_jefe     0.172182
conex_inte    0.136582
educa_jefe    0.090725
tenencia      0.083007
tam_loc       0.033798
dtype: float64

Bathrooms

Numerical predictors - Spearman
ing_cor      0.411836
num_auto     0.316685
edad_jefe    0.064726
num_moto    -0.056331
num_bici    -0.046002
tot_integ   -0.036726
Name: banos_completos, dtype: float64

Categorical predictors - Cramer's V
conex_inte    0.374200
educa_jefe    0.268710
tam_loc       0.255672
tenencia      0.156719
sexo_jefe     0.016919
dtype: float64

Rooms

Numerical predictors - Spearman
tot_integ    0.389040
ing_cor      0.370281
num_auto     0.204063
edad_jefe    0.127844
num_bici     0.041350
num_moto     0.031076
Name: cuartos_dormir, dtype: float64

Categorical predictors - C

The association patterns differ across the three target variables, so a single feature set is not imposed on every model.

For each target, we define two levels of predictor information:

- **Base features:** variables selected from their statistical association with the target together with their direct household interpretation.
- **Extended features:** the base variables plus additional harmonized household characteristics that may provide complementary predictive information.

We evaluate both alternatives empirically.

Some variables examined above, such as `tam_loc`, are not included in the final feature sets because the model can only use predictors with a compatible representation in both ENIGH and the OD survey.

### 2.3 Harmonization of model predictors

Before training the models, the predictor definitions in ENIGH and the OD survey must be expressed in the same representation.

The harmonized variables are:

1. `personas_en_vivienda`
2. `ingreso_hogar`
3. `educacion_jefe`
4. `tiene_internet`
5. `n_autos_camionetas`
6. `n_motos`
7. `n_bicicletas`
8. `tenencia_vivienda`
9. `edad_jefe`
10. `sexo_jefe`

Household income requires an additional transformation because ENIGH reports quarterly current income whereas the OD survey uses monthly income categories. For ENIGH household $h$,

$$I_h^{(\mathrm{monthly})} = \frac{I_h^{(\mathrm{quarterly})}}{3}.$$

The resulting monthly value is then assigned to the same income intervals used by the OD survey.

The head-of-household education predictor is represented by the AMAI education score. For OD dwellings, this variable corresponds to the observed or probability-weighted education score constructed in the previous notebook.

Categorical variables are subsequently mapped so that equivalent concepts use the same labels in both datasets.

In [ ]:
###################################
# Household-size harmonization
###################################

enigh_households["personas_en_vivienda"] = enigh_households["tot_integ"].apply(lambda x: "10 y +" if x >= 10 else str(x)).astype("string")
od_housing["personas_en_vivienda"] = od_housing["personas_en_vivienda"].astype("string")

############################
# ingreso_hogar homologation
############################

# Convert ENIGH quarterly current household income to monthly income
enigh_households["ingreso_mensual_hogar"] = enigh_households["ing_cor"] / 3

# Map ENIGH monthly income to the income categories available in the OD survey
enigh_households["ingreso_hogar"] = pd.NA
enigh_households.loc[enigh_households["ingreso_mensual_hogar"] == 0, "ingreso_hogar"] = "Sin ingresos"
enigh_households.loc[(enigh_households["ingreso_mensual_hogar"] > 0) & (enigh_households["ingreso_mensual_hogar"] <= 1500), "ingreso_hogar"] = "Hasta $1,500"
enigh_households.loc[(enigh_households["ingreso_mensual_hogar"] > 1500) & (enigh_households["ingreso_mensual_hogar"] <= 3000), "ingreso_hogar"] = "$1,501 y $3,000"
enigh_households.loc[(enigh_households["ingreso_mensual_hogar"] > 3000) & (enigh_households["ingreso_mensual_hogar"] <= 7000), "ingreso_hogar"] = "$3,001 y $7,000"
enigh_households.loc[(enigh_households["ingreso_mensual_hogar"] > 7000) & (enigh_households["ingreso_mensual_hogar"] <= 10000), "ingreso_hogar"] = "$7,001 y $10,000"
enigh_households.loc[(enigh_households["ingreso_mensual_hogar"] > 10000) & (enigh_households["ingreso_mensual_hogar"] <= 15000), "ingreso_hogar"] = "$10,001 y $15,000"
enigh_households.loc[(enigh_households["ingreso_mensual_hogar"] > 15000) & (enigh_households["ingreso_mensual_hogar"] <= 25000), "ingreso_hogar"] = "$15,001 y $25,000"
enigh_households.loc[enigh_households["ingreso_mensual_hogar"] > 25000, "ingreso_hogar"] = "Más de $25,001"
enigh_households["ingreso_hogar"] = enigh_households["ingreso_hogar"].astype("string")

# Treat non-response income categories in the OD survey as missing
od_housing["ingreso_hogar"] = od_housing["ingreso_mensual_hogar"].replace({
    "No quiso responder": pd.NA,
    "No sabe": pd.NA
}).astype("string")

#############################
# educacion_jefe homologation
#############################

# Represent ENIGH head-of-household education using the corresponding AMAI score
enigh_education_score_mapping = {
    "01": 0,
    "02": 0,
    "03": 6,
    "04": 11,
    "05": 12,
    "06": 18,
    "07": 23,
    "08": 27,
    "09": 36,
    "10": 59,
    "11": 85
}

enigh_households["educacion_jefe"] = enigh_households["educa_jefe"].str.strip().map(enigh_education_score_mapping).astype("Float64")
# Use the observed or probability-weighted education score generated in the previous notebook
od_housing["educacion_jefe"] = od_housing["modelo_puntos_ultimo_estudio_jefe_hogar"].astype("Float64")

#############################
# tiene_internet homologation
#############################

internet_mapping = {
    "1": "Sí",
    "2": "No"
}

# Harmonize the ENIGH Internet-access codes with the OD labels
enigh_households["tiene_internet"] = enigh_households["conex_inte"].str.strip().map(internet_mapping).astype("string")
od_housing["tiene_internet"] = od_housing["tiene_internet"].astype("string")

#######################################################
# n_autos_camionetas, n_motos, n_bicicletas homologation
#######################################################

vehicle_count_mapping = {
    "0": "0",
    "1": "1",
    "2": "2",
    "3": "3",
    "4": "4 o más",
    "5": "4 o más",
    "6": "4 o más",
    "7": "4 o más",
    "8": "4 o más",
    "9": "4 o más",
    "10": "4 o más",
    "11": "4 o más",
    "12": "4 o más"
}

# Collapse ENIGH vehicle counts to the categories available in the OD survey
enigh_households["n_autos_camionetas"] = enigh_households["num_auto"].map(vehicle_count_mapping).astype("string")
enigh_households["n_motos"] = enigh_households["num_moto"].map(vehicle_count_mapping).astype("string")
enigh_households["n_bicicletas"] = enigh_households["num_bici"].map(vehicle_count_mapping).astype("string")

od_housing["n_autos_camionetas"] = od_housing["n_autos_camionetas"].astype("string")
od_housing["n_motos"] = od_housing["n_motos"].astype("string")
od_housing["n_bicicletas"] = od_housing["n_bicicletas"].astype("string")

################################
# tenencia_vivienda homologation
################################

# Map ENIGH housing-tenure codes to the categories used for harmonization
tenure_mapping = {
    "1": "Rentada",
    "2": "Prestada",
    "3": "Hipotecada",
    "4": "Propia",
    "5": "Otros",
    "6": "Otros"
}

enigh_housing["tenencia_vivienda"] = enigh_housing["tenencia"].map(tenure_mapping).astype("string")

# Obtain one housing-tenure value per dwelling
housing_tenure = enigh_housing.reset_index()[["folioviv", "tenencia_vivienda"]].copy()

# Attach dwelling-level tenure to every ENIGH household in the dwelling
enigh_households = enigh_households.merge(housing_tenure, on="folioviv", how="left", validate="many_to_one")

od_housing["tenencia_vivienda"] = od_housing["tenencia_vivienda"].astype("string")

###################################
# edad_jefe, sexo_jefe homologation
###################################

# Obtain age and sex from the head-of-household records constructed in the previous notebook
head_demographics = od_population.loc[od_population["jefe_hogar_asignacion_artificial"] == "Jefe del hogar", ["folio_vivienda", "edad", "sexo_nacimiento"]].copy()

# Reduce the head-of-household information to one record per dwelling
head_demographics = head_demographics.groupby("folio_vivienda", as_index=False)[["edad", "sexo_nacimiento"]].first()

# Rename the variables to explicitly identify them as head-of-household attributes
head_demographics = head_demographics.rename(columns={"edad": "edad_jefe", "sexo_nacimiento": "sexo_jefe"})

# Attach head-of-household age and sex to the OD dwelling table
od_housing = od_housing.merge(head_demographics, on="folio_vivienda", how="left", validate="one_to_one")

enigh_households["edad_jefe"] = enigh_households["edad_jefe"].astype("Int64")
od_housing["edad_jefe"] = od_housing["edad_jefe"].astype("Int64")

sex_mapping = {
    "1": "Hombres",
    "2": "Mujeres"
}

enigh_households["sexo_jefe"] = enigh_households["sexo_jefe"].map(sex_mapping).astype("string")
od_housing["sexo_jefe"] = od_housing["sexo_jefe"].astype("string")

### 2.4 Candidate feature sets

The previous transformations provide a common predictor space for ENIGH and OD.

For each target, four candidate feature sets are constructed:

1. Base with income.
2. Extended with income.
3. Base without income.
4. Extended without income.

The duplicated structure with and without income is intentional. It allows the same target to be modeled under two information scenarios and avoids discarding OD dwellings for which household income was not reported.

In [ ]:
# Define Base and Extended feature sets for each target, with and without household income
workers_base_features = [
    "personas_en_vivienda",
    "ingreso_hogar"
]

workers_extended_features = workers_base_features + [
    "sexo_jefe",
    "edad_jefe",
    "tiene_internet",
    "n_autos_camionetas",
    "n_motos",
    "n_bicicletas",
    "educacion_jefe",
    "tenencia_vivienda"
]

workers_base_without_income_features = [
    "personas_en_vivienda"
]

workers_extended_without_income_features = [
    "personas_en_vivienda",
    "sexo_jefe",
    "edad_jefe",
    "tiene_internet",
    "n_autos_camionetas",
    "n_motos",
    "n_bicicletas",
    "educacion_jefe",
    "tenencia_vivienda"
]

bathrooms_base_features = [
    "ingreso_hogar",
    "tiene_internet",
    "n_autos_camionetas",
    "educacion_jefe",
    "tenencia_vivienda"
]

bathrooms_extended_features = bathrooms_base_features + [
    "personas_en_vivienda",
    "edad_jefe",
    "n_motos",
    "n_bicicletas"
]

bathrooms_base_without_income_features = [
    "tiene_internet",
    "n_autos_camionetas",
    "educacion_jefe",
    "tenencia_vivienda"
]

bathrooms_extended_without_income_features = [
    "tiene_internet",
    "n_autos_camionetas",
    "educacion_jefe",
    "tenencia_vivienda",
    "personas_en_vivienda",
    "edad_jefe",
    "n_motos",
    "n_bicicletas"
]

rooms_base_features = [
    "personas_en_vivienda",
    "ingreso_hogar",
    "tiene_internet",
    "n_autos_camionetas",
    "tenencia_vivienda"
]

rooms_extended_features = rooms_base_features + [
    "edad_jefe",
    "educacion_jefe",
    "n_motos",
    "n_bicicletas"
]

rooms_base_without_income_features = [
    "personas_en_vivienda",
    "tiene_internet",
    "n_autos_camionetas",
    "tenencia_vivienda"
]

rooms_extended_without_income_features = [
    "personas_en_vivienda",
    "tiene_internet",
    "n_autos_camionetas",
    "tenencia_vivienda",
    "edad_jefe",
    "educacion_jefe",
    "n_motos",
    "n_bicicletas"
]

workers_feature_sets = {
    "Base with income": workers_base_features,
    "Extended with income": workers_extended_features,
    "Base without income": workers_base_without_income_features,
    "Extended without income": workers_extended_without_income_features
}

bathrooms_feature_sets = {
    "Base with income": bathrooms_base_features,
    "Extended with income": bathrooms_extended_features,
    "Base without income": bathrooms_base_without_income_features,
    "Extended without income": bathrooms_extended_without_income_features
}

rooms_feature_sets = {
    "Base with income": rooms_base_features,
    "Extended with income": rooms_extended_features,
    "Base without income": rooms_base_without_income_features,
    "Extended without income": rooms_extended_without_income_features
}

### 2.5 Missingness in the harmonized predictors

Before fitting the models, we compare predictor availability in ENIGH and the OD survey.

The same diagnostic is calculated for each complete candidate feature set.

In [ ]:
harmonized_features = [
    "personas_en_vivienda",
    "ingreso_hogar",
    "educacion_jefe",
    "tiene_internet",
    "n_autos_camionetas",
    "n_motos",
    "n_bicicletas",
    "tenencia_vivienda",
    "edad_jefe",
    "sexo_jefe"
]

# Compare predictor availability in the ENIGH training domain and the OD application domain
print("ENIGH missingness")
print(enigh_households[harmonized_features].isna().mean().mul(100).sort_values(ascending=False))

print("\nOD missingness")
print(od_housing[harmonized_features].isna().mean().mul(100).sort_values(ascending=False))

print()
for feature in harmonized_features:
    enigh_missing = enigh_households[feature].isna().mean() * 100
    od_missing = od_housing[feature].isna().mean() * 100
    print(f"{feature}: ENIGH={enigh_missing:.2f}%, OD={od_missing:.2f}%")

feature_sets = {
    "Workers": workers_feature_sets,
    "Bathrooms": bathrooms_feature_sets,
    "Rooms": rooms_feature_sets
}

for target_name, target_feature_sets in feature_sets.items():
    print(f"\n{'=' * 60}")
    print(target_name)
    print("=" * 60)

    # Repeat the missingness diagnostic for each candidate model feature set
    for feature_set_name, features in target_feature_sets.items():
        print(f"\n{feature_set_name}")

        missingness = pd.DataFrame({
            "ENIGH": enigh_households[features].isna().mean() * 100,
            "OD": od_housing[features].isna().mean() * 100
        })

        print(missingness.sort_values("OD", ascending=False))

ENIGH missingness
personas_en_vivienda    0.0
ingreso_hogar           0.0
educacion_jefe          0.0
tiene_internet          0.0
n_autos_camionetas      0.0
n_motos                 0.0
n_bicicletas            0.0
tenencia_vivienda       0.0
edad_jefe               0.0
sexo_jefe               0.0
dtype: float64

OD missingness
ingreso_hogar           58.276074
personas_en_vivienda     0.000000
educacion_jefe           0.000000
tiene_internet           0.000000
n_autos_camionetas       0.000000
n_motos                  0.000000
n_bicicletas             0.000000
tenencia_vivienda        0.000000
edad_jefe                0.000000
sexo_jefe                0.000000
dtype: float64

personas_en_vivienda: ENIGH=0.00%, OD=0.00%
ingreso_hogar: ENIGH=0.00%, OD=58.28%
educacion_jefe: ENIGH=0.00%, OD=0.00%
tiene_internet: ENIGH=0.00%, OD=0.00%
n_autos_camionetas: ENIGH=0.00%, OD=0.00%
n_motos: ENIGH=0.00%, OD=0.00%
n_bicicletas: ENIGH=0.00%, OD=0.00%
tenencia_vivienda: ENIGH=0.00%, OD=0.00%
edad_je

The missingness analysis identifies household income as the relevant difference between the two datasets.

All harmonized predictors used by the models are complete in ENIGH. In the OD survey, however, `ingreso_hogar` is missing for approximately 58.28% of dwellings.

A model that always requires income could therefore only be applied to approximately 42% of the OD observations.

For this reason, we use a hybrid prediction strategy. For each target $t$, two models are estimated: $f_t^{(+I)}$ using the feature set that includes household income, and $f_t^{(-I)}$ using an otherwise comparable feature set that excludes income.

When the models are later applied to OD dwelling $h$, the prediction rule is

$$\widehat{\mathbf p}_h^{(t)} = \begin{cases} f_t^{(+I)}(X_h^{(+I)}), & \text{if household income is observed},\\[4pt] f_t^{(-I)}(X_h^{(-I)}), & \text{if household income is missing}. \end{cases}$$

This allows all OD dwellings to receive predictions while retaining the information contained in household income whenever it is available.

## 3. Model training and evaluation

### 3.1 Training and test partitions

The split is performed independently for Workers, Bathrooms, and Rooms because the availability of the target variable is not the same across the three datasets.

Multiple households can belong to the same ENIGH dwelling identified by `folioviv`. To prevent information from the same dwelling from appearing in both partitions, the split is grouped by `folioviv`.

For each target $t$,

$$\mathcal{D}^{(t)} = \mathcal{D}_{\mathrm{train}}^{(t)} \cup \mathcal{D}_{\mathrm{test}}^{(t)},$$

with

$$\mathcal{G}_{\mathrm{train}}^{(t)} \cap \mathcal{G}_{\mathrm {test}}^{(t)} = \varnothing,$$

where $\mathcal{G}$ is the set of `folioviv` identifiers.

Stratification is used to approximately preserve the target-category distribution. One of five grouped folds is reserved for testing, resulting in an approximately 80/20 partition.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
target_columns = {
    "Workers": "amai_14_anos_trabajo_mes",
    "Bathrooms": "amai_banos_completos",
    "Rooms": "amai_cuartos_dormir"
}

# Restrict each modeling dataset to observations with a known target category
model_data = {
    "Workers": enigh_households.loc[enigh_households["amai_14_anos_trabajo_mes"].notna()].copy(),
    "Bathrooms": enigh_households.loc[enigh_households["amai_banos_completos"].notna()].copy(),
    "Rooms": enigh_households.loc[enigh_households["amai_cuartos_dormir"].notna()].copy()
}

# Reserve one grouped and stratified fold as the final test partition
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_data = {}
test_data = {}

for target_name, target_column in target_columns.items():
    data = model_data[target_name]
    # Split independently by target because target availability differs across components
    train_idx, test_idx = next(splitter.split(data, data[target_column], groups=data["folioviv"]))
    train_data[target_name] = data.iloc[train_idx].copy()
    test_data[target_name] = data.iloc[test_idx].copy()

    print(f"\n{'=' * 60}")
    print(target_name)
    print("=" * 60)
    print(f"Training observations: {len(train_data[target_name]):,}")
    print(f"Testing observations: {len(test_data[target_name]):,}")
    print(f"Training dwellings: {train_data[target_name]['folioviv'].nunique():,}")
    print(f"Testing dwellings: {test_data[target_name]['folioviv'].nunique():,}")
    print(f"Dwelling overlap: {len(set(train_data[target_name]['folioviv']) & set(test_data[target_name]['folioviv']))}")


Workers
Training observations: 72,081
Testing observations: 18,021
Training dwellings: 71,059
Testing dwellings: 17,764
Dwelling overlap: 0

Bathrooms
Training observations: 70,691
Testing observations: 17,673
Training dwellings: 69,676
Testing dwellings: 17,418
Dwelling overlap: 0

Rooms
Training observations: 72,081
Testing observations: 18,021
Training dwellings: 71,057
Testing dwellings: 17,766
Dwelling overlap: 0


In [ ]:
# Verify that target-category shares remain comparable after the grouped train/test split
split_distributions = {}
for target_name, target_column in target_columns.items():
    data = model_data[target_name]
    train = train_data[target_name]
    test = test_data[target_name]

    original_distribution = data[target_column].value_counts(normalize=True).rename("original")
    train_distribution = train[target_column].value_counts(normalize=True).rename("train")
    test_distribution = test[target_column].value_counts(normalize=True).rename("test")

    split_distributions[target_name] = pd.concat([original_distribution, train_distribution, test_distribution], axis=1).mul(100).round(2)

    print(f"\n{'=' * 60}")
    print(target_name)
    print("=" * 60)
    print(split_distributions[target_name])


Workers
                          original  train   test
amai_14_anos_trabajo_mes                        
1                            38.32  38.32  38.32
2                            32.59  32.59  32.59
3                            12.01  12.01  12.01
0                            11.22  11.22  11.22
4 o más                       5.85   5.85   5.85

Bathrooms
                      original  train   test
amai_banos_completos                        
1                        58.66  58.66  58.65
0                        24.92  24.92  24.93
2 o más                  16.42  16.42  16.42

Rooms
                     original  train   test
amai_cuartos_dormir                        
2                       43.13  43.13  43.13
1                       28.28  28.28  28.28
3                       22.44  22.44  22.44
4 o más                  6.15   6.15   6.15


The resulting partitions preserve the original category distributions while maintaining zero overlap in `folioviv` between training and testing.

The target shares are nearly identical in the complete, training, and test samples for all three components. We can therefore proceed with model selection without introducing a material change in the target distribution through the split.

### 3.2 Model and feature-set benchmark

We first compare model families and feature sets using the training partition only.

Three probabilistic classifiers are considered:

- **Logistic Regression**, as a linear probabilistic baseline.
- **Random Forest**, to represent nonlinear relationships and interactions through an ensemble of decision trees.
- **Gradient Boosting**, to represent nonlinear relationships through sequentially fitted trees.

For each AMAI target, the models are evaluated separately under the with-income and without-income scenarios.

Five-fold stratified grouped cross-validation is used inside the training sample. ENIGH expansion factors are incorporated during both model fitting and metric calculation.

The primary evaluation metric is the survey-weighted multiclass log loss:

$$\mathcal{L}^{(t)} = - \frac{1}{\sum_h w_h} \sum_h w_h \sum_{k=1}^{K_t} \mathbf{1} \left(Y_h^{(t)}=k\right) \log \left(\hat p_{hk}^{(t)} \right),$$

where $K_t$ is the number of categories for target $t$.

Weighted accuracy is also reported:

$$A^{(t)} = \frac{\sum_h w_h \mathbf{1}\left(\widehat Y_h^{(t)} = Y_h^{(t)}\right)}{\sum_h w_h}.$$

Log loss is used as the primary selection criterion because the downstream AMAI calculation uses the complete predicted probability distribution rather than only the most probable category.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

In [ ]:
weight_column = "factor"

model_feature_sets = {
    "Workers": workers_feature_sets,
    "Bathrooms": bathrooms_feature_sets,
    "Rooms": rooms_feature_sets
}

# Initial model configurations used for the benchmark stage
models = {
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
}

def build_pipeline(model_name, model, features):
    # Treat head age and education score as numerical; remaining harmonized features are categorical
    numeric_model_features = [feature for feature in features if feature in ["edad_jefe", "educacion_jefe"]]
    # Gradient Boosting requires a dense one-hot encoded matrix
    categorical_model_features = [feature for feature in features if feature not in numeric_model_features]
    numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=model_name != "Gradient Boosting"))])
    preprocessor = ColumnTransformer([("numeric", numeric_transformer, numeric_model_features), ("categorical", categorical_transformer, categorical_model_features)])
    return Pipeline([("preprocessor", preprocessor), ("model", clone(model))])

In [22]:
from tqdm.auto import tqdm

In [ ]:
# Compare every model–feature-set combination using grouped cross-validation within the training data
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
benchmark_results = []
total_fits = sum(len(feature_sets) * len(models) * cv.get_n_splits() for feature_sets in model_feature_sets.values())
with tqdm(total=total_fits, desc="Benchmark progress") as progress_bar:
    for target_name, target_column in target_columns.items():
        target_train = train_data[target_name]
        target_feature_sets = model_feature_sets[target_name]

        tqdm.write(f"\n{'=' * 60}")
        tqdm.write(f"{target_name}")
        tqdm.write("=" * 60)

        for feature_set_name, features in target_feature_sets.items():
            income_scenario = "Without income" if "without income" in feature_set_name.lower() else "With income"

            X = target_train[features]
            y = target_train[target_column]
            groups = target_train["folioviv"]
            weights = target_train[weight_column]

            tqdm.write(f"\nFeature set: {feature_set_name}")

            for model_name, model in models.items():
                fold_log_losses = []
                fold_accuracies = []

                tqdm.write(f"Model: {model_name}")

                for fold, (train_fold_idx, val_fold_idx) in enumerate(cv.split(X, y, groups), start=1):
                    X_train_fold = X.iloc[train_fold_idx]
                    X_val_fold = X.iloc[val_fold_idx]
                    y_train_fold = y.iloc[train_fold_idx]
                    y_val_fold = y.iloc[val_fold_idx]
                    w_train_fold = weights.iloc[train_fold_idx]
                    w_val_fold = weights.iloc[val_fold_idx]

                    # Use ENIGH expansion factors as sample weights during model fitting
                    pipeline = build_pipeline(model_name, model, features)
                    pipeline.fit(X_train_fold, y_train_fold, model__sample_weight=w_train_fold)

                    val_probabilities = pipeline.predict_proba(X_val_fold)
                    val_predictions = pipeline.predict(X_val_fold)
                    classes = pipeline.named_steps["model"].classes_

                    # Evaluate probabilistic and class predictions using the same survey weights
                    fold_log_loss = log_loss(y_val_fold, val_probabilities, labels=classes, sample_weight=w_val_fold)
                    fold_accuracy = accuracy_score(y_val_fold, val_predictions, sample_weight=w_val_fold)

                    fold_log_losses.append(fold_log_loss)
                    fold_accuracies.append(fold_accuracy)

                    progress_bar.set_postfix(target=target_name, model=model_name, fold=f"{fold}/5", log_loss=f"{fold_log_loss:.3f}")
                    progress_bar.update(1)

                benchmark_results.append({
                    "target": target_name,
                    "income_scenario": income_scenario,
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "log_loss_mean": np.mean(fold_log_losses),
                    "log_loss_std": np.std(fold_log_losses),
                    "accuracy_mean": np.mean(fold_accuracies),
                    "accuracy_std": np.std(fold_accuracies)
                })

                tqdm.write(f"  Log loss: {np.mean(fold_log_losses):.4f} ± {np.std(fold_log_losses):.4f}")
                tqdm.write(f"  Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")

Benchmark progress:   0%|          | 0/180 [00:00<?, ?it/s]


Workers

Feature set: Base with income
Model: Logistic Regression
  Log loss: 1.1081 ± 0.0055
  Accuracy: 0.4820 ± 0.0044
Model: Random Forest
  Log loss: 1.1098 ± 0.0047
  Accuracy: 0.4807 ± 0.0041
Model: Gradient Boosting
  Log loss: 1.1143 ± 0.0051
  Accuracy: 0.4817 ± 0.0038

Feature set: Extended with income
Model: Logistic Regression
  Log loss: 1.0491 ± 0.0040
  Accuracy: 0.5088 ± 0.0050
Model: Random Forest
  Log loss: 1.0202 ± 0.0024
  Accuracy: 0.5301 ± 0.0047
Model: Gradient Boosting
  Log loss: 1.0207 ± 0.0022
  Accuracy: 0.5286 ± 0.0028

Feature set: Base without income
Model: Logistic Regression
  Log loss: 1.1387 ± 0.0038
  Accuracy: 0.4459 ± 0.0067
Model: Random Forest
  Log loss: 1.1386 ± 0.0038
  Accuracy: 0.4459 ± 0.0067
Model: Gradient Boosting
  Log loss: 1.1422 ± 0.0038
  Accuracy: 0.4454 ± 0.0066

Feature set: Extended without income
Model: Logistic Regression
  Log loss: 1.0817 ± 0.0024
  Accuracy: 0.4806 ± 0.0025
Model: Random Forest
  Log loss: 1.0471 ± 0.004

In [26]:
benchmark_results = pd.DataFrame(benchmark_results).sort_values(["target", "income_scenario", "log_loss_mean", "accuracy_mean"], ascending=[True, True, True, False]).reset_index(drop=True)
benchmark_results

,target,income_scenario,feature_set,model,log_loss_mean,log_loss_std,accuracy_mean,accuracy_std
0,Bathrooms,With income,Extended with income,Logistic Regression,0.751582,0.008749,0.657920,0.005686
1,Bathrooms,With income,Extended with income,Gradient Boosting,0.753789,0.007018,0.657181,0.005130
2,Bathrooms,With income,Extended with income,Random Forest,0.759437,0.007506,0.652643,0.003677
3,Bathrooms,With income,Base with income,Logistic Regression,0.781609,0.008837,0.635164,0.003791
4,Bathrooms,With income,Base with income,Gradient Boosting,0.781903,0.008376,0.636228,0.004050
5,Bathrooms,With income,Base with income,Random Forest,0.789196,0.007306,0.633401,0.003845
6,Bathrooms,Without income,Extended without income,Gradient Boosting,0.777108,0.008434,0.641147,0.006538
7,Bathrooms,Without income,Extended without income,Logistic Regression,0.779648,0.009288,0.640524,0.007116
8,Bathrooms,Without income,Extended without income,Random Forest,0.786191,0.008869,0.634946,0.004241
9,Bathrooms,Without income,Base without income,Gradient Boosting,0.804290,0.009060,0.618980,0.005841


In [ ]:
# Select the lowest-log-loss model and feature set independently for each target and income scenario
best_benchmark_results = benchmark_results.groupby(["target", "income_scenario"], as_index=False).first()
best_benchmark_results

,target,income_scenario,feature_set,model,log_loss_mean,log_loss_std,accuracy_mean,accuracy_std
0,Bathrooms,With income,Extended with income,Logistic Regression,0.751582,0.008749,0.657920,0.005686
1,Bathrooms,Without income,Extended without income,Gradient Boosting,0.777108,0.008434,0.641147,0.006538
2,Rooms,With income,Extended with income,Gradient Boosting,1.031491,0.004532,0.536198,0.004193
3,Rooms,Without income,Extended without income,Gradient Boosting,1.043471,0.003674,0.530267,0.003579
4,Workers,With income,Extended with income,Random Forest,1.020196,0.002374,0.530075,0.004710
5,Workers,Without income,Extended without income,Random Forest,1.047071,0.004750,0.497727,0.006186


In [28]:
for _, result in best_benchmark_results.iterrows():
    print(f"\n{'=' * 60}")
    print(f"{result['target']} - {result['income_scenario']}")
    print("=" * 60)
    print(f"Best feature set: {result['feature_set']}")
    print(f"Best model: {result['model']}")
    print(f"CV weighted log loss: {result['log_loss_mean']:.4f} ± {result['log_loss_std']:.4f}")
    print(f"CV weighted accuracy: {result['accuracy_mean']:.4f} ± {result['accuracy_std']:.4f}")


Bathrooms - With income
Best feature set: Extended with income
Best model: Logistic Regression
CV weighted log loss: 0.7516 ± 0.0087
CV weighted accuracy: 0.6579 ± 0.0057

Bathrooms - Without income
Best feature set: Extended without income
Best model: Gradient Boosting
CV weighted log loss: 0.7771 ± 0.0084
CV weighted accuracy: 0.6411 ± 0.0065

Rooms - With income
Best feature set: Extended with income
Best model: Gradient Boosting
CV weighted log loss: 1.0315 ± 0.0045
CV weighted accuracy: 0.5362 ± 0.0042

Rooms - Without income
Best feature set: Extended without income
Best model: Gradient Boosting
CV weighted log loss: 1.0435 ± 0.0037
CV weighted accuracy: 0.5303 ± 0.0036

Workers - With income
Best feature set: Extended with income
Best model: Random Forest
CV weighted log loss: 1.0202 ± 0.0024
CV weighted accuracy: 0.5301 ± 0.0047

Workers - Without income
Best feature set: Extended without income
Best model: Random Forest
CV weighted log loss: 1.0471 ± 0.0048
CV weighted accura

The benchmark selects an Extended feature set in all six target–income combinations.

The selected model families are:

| Target | With income | Without income |
| --- | --- | --- |
| Workers | Random Forest | Random Forest |
| Bathrooms | Logistic Regression | Gradient Boosting |
| Rooms | Gradient Boosting | Gradient Boosting |

The result also shows that including the additional harmonized predictors improves the probabilistic performance relative to the corresponding Base feature sets.

Having selected one model family and feature set for each scenario, the next stage tunes only the corresponding model hyperparameters.

### 3.3 Hyperparameter tuning

Model family and feature-set selection are now fixed from the benchmark stage.

For each of the six selected target–income combinations, a predefined grid of hyperparameters is evaluated using the same five-fold stratified grouped cross-validation procedure.

The hyperparameter grids are:

- **Logistic Regression:** regularization parameter $C$.
- **Random Forest:** number of trees, minimum leaf size, maximum depth, and number of features considered at each split.
- **Gradient Boosting:** number of boosting stages, learning rate, and maximum tree depth.

As in the benchmark, configurations are ranked primarily by minimum weighted log loss and secondarily by weighted accuracy.

In [ ]:
# Store the benchmark winner for each target and income scenario before hyperparameter tuning
best_benchmark_results = benchmark_results.sort_values(["target", "income_scenario", "log_loss_mean", "accuracy_mean"], ascending=[True, True, True, False]).groupby(["target", "income_scenario"], as_index=False).first()
best_configurations = {}
for _, result in best_benchmark_results.iterrows():
    target_name = result["target"]
    income_scenario = result["income_scenario"]

    if target_name not in best_configurations:
        best_configurations[target_name] = {}

    best_configurations[target_name][income_scenario] = {
        "feature_set": result["feature_set"],
        "features": model_feature_sets[target_name][result["feature_set"]],
        "model": result["model"],
        "log_loss": result["log_loss_mean"],
        "accuracy": result["accuracy_mean"]
    }

In [ ]:
# Candidate hyperparameters for the model families that may be selected by the benchmark
parameter_grids = {
    "Logistic Regression": [
        {"C": 0.1},
        {"C": 0.5},
        {"C": 1.0},
        {"C": 5.0},
        {"C": 10.0}
    ],

    "Random Forest": [
        {"n_estimators": 300, "min_samples_leaf": 2, "max_depth": None, "max_features": "sqrt"},
        {"n_estimators": 300, "min_samples_leaf": 5, "max_depth": None, "max_features": "sqrt"},
        {"n_estimators": 500, "min_samples_leaf": 5, "max_depth": None, "max_features": "sqrt"},
        {"n_estimators": 500, "min_samples_leaf": 10, "max_depth": None, "max_features": "sqrt"},
        {"n_estimators": 500, "min_samples_leaf": 5, "max_depth": 20, "max_features": "sqrt"},
        {"n_estimators": 500, "min_samples_leaf": 10, "max_depth": 20, "max_features": 0.7}
    ],

    "Gradient Boosting": [
        {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2},
        {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 2},
        {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 3},
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
        {"n_estimators": 150, "learning_rate": 0.1, "max_depth": 2},
        {"n_estimators": 200, "learning_rate": 0.1, "max_depth": 2}
    ]
}

def make_model(model_name, params):
    if model_name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, random_state=42, **params)

    if model_name == "Random Forest":
        return RandomForestClassifier(random_state=42, n_jobs=-1, **params)

    if model_name == "Gradient Boosting":
        return GradientBoostingClassifier(random_state=42, **params)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Tune only the model family and feature set selected for each target–scenario combination
tuning_results = []
total_fits = sum(len(parameter_grids[best_configurations[target_name][income_scenario]["model"]]) * cv.get_n_splits() for target_name in best_configurations for income_scenario in best_configurations[target_name])
with tqdm(total=total_fits, desc="Tuning progress") as progress_bar:
    for target_name, target_column in target_columns.items():
        target_train = train_data[target_name]

        for income_scenario, configuration in best_configurations[target_name].items():
            model_name = configuration["model"]
            feature_set_name = configuration["feature_set"]
            features = configuration["features"]

            X = target_train[features]
            y = target_train[target_column]
            groups = target_train["folioviv"]
            weights = target_train[weight_column]

            tqdm.write(f"\n{'=' * 60}")
            tqdm.write(f"{target_name} - {income_scenario}")
            tqdm.write("=" * 60)
            tqdm.write(f"Model: {model_name}")
            tqdm.write(f"Feature set: {feature_set_name}")

            for params in parameter_grids[model_name]:
                fold_log_losses = []
                fold_accuracies = []

                tqdm.write(f"\nParameters: {params}")

                for fold, (train_fold_idx, val_fold_idx) in enumerate(cv.split(X, y, groups), start=1):
                    X_train_fold = X.iloc[train_fold_idx]
                    X_val_fold = X.iloc[val_fold_idx]
                    y_train_fold = y.iloc[train_fold_idx]
                    y_val_fold = y.iloc[val_fold_idx]
                    w_train_fold = weights.iloc[train_fold_idx]
                    w_val_fold = weights.iloc[val_fold_idx]

                    model = make_model(model_name, params)
                    pipeline = build_pipeline(model_name, model, features)

                    # Keep dwelling groups separated within every cross-validation fold
                    pipeline.fit(X_train_fold, y_train_fold, model__sample_weight=w_train_fold)

                    val_probabilities = pipeline.predict_proba(X_val_fold)
                    val_predictions = pipeline.predict(X_val_fold)
                    classes = pipeline.named_steps["model"].classes_

                    fold_log_loss = log_loss(y_val_fold, val_probabilities, labels=classes, sample_weight=w_val_fold)
                    fold_accuracy = accuracy_score(y_val_fold, val_predictions, sample_weight=w_val_fold)

                    fold_log_losses.append(fold_log_loss)
                    fold_accuracies.append(fold_accuracy)

                    progress_bar.set_postfix(target=target_name, scenario=income_scenario, fold=f"{fold}/5", log_loss=f"{fold_log_loss:.3f}")
                    progress_bar.update(1)

                tuning_results.append({
                    "target": target_name,
                    "income_scenario": income_scenario,
                    "model": model_name,
                    "feature_set": feature_set_name,
                    "params": params,
                    "log_loss_mean": np.mean(fold_log_losses),
                    "log_loss_std": np.std(fold_log_losses),
                    "accuracy_mean": np.mean(fold_accuracies),
                    "accuracy_std": np.std(fold_accuracies)
                })

                tqdm.write(f"  Log loss: {np.mean(fold_log_losses):.4f} ± {np.std(fold_log_losses):.4f}")
                tqdm.write(f"  Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")

Tuning progress:   0%|          | 0/175 [00:00<?, ?it/s]


Workers - With income
Model: Random Forest
Feature set: Extended with income

Parameters: {'n_estimators': 300, 'min_samples_leaf': 2, 'max_depth': None, 'max_features': 'sqrt'}
  Log loss: 1.0420 ± 0.0045
  Accuracy: 0.5073 ± 0.0060

Parameters: {'n_estimators': 300, 'min_samples_leaf': 5, 'max_depth': None, 'max_features': 'sqrt'}
  Log loss: 1.0202 ± 0.0024
  Accuracy: 0.5301 ± 0.0047

Parameters: {'n_estimators': 500, 'min_samples_leaf': 5, 'max_depth': None, 'max_features': 'sqrt'}
  Log loss: 1.0197 ± 0.0026
  Accuracy: 0.5291 ± 0.0031

Parameters: {'n_estimators': 500, 'min_samples_leaf': 10, 'max_depth': None, 'max_features': 'sqrt'}
  Log loss: 1.0229 ± 0.0024
  Accuracy: 0.5312 ± 0.0041

Parameters: {'n_estimators': 500, 'min_samples_leaf': 5, 'max_depth': 20, 'max_features': 'sqrt'}
  Log loss: 1.0189 ± 0.0025
  Accuracy: 0.5317 ± 0.0035

Parameters: {'n_estimators': 500, 'min_samples_leaf': 10, 'max_depth': 20, 'max_features': 0.7}
  Log loss: 1.0214 ± 0.0061
  Accuracy: 0

In [33]:
tuning_results = pd.DataFrame(tuning_results).sort_values(["target", "income_scenario", "log_loss_mean", "accuracy_mean"], ascending=[True, True, True, False]).reset_index(drop=True)
tuning_results

,target,income_scenario,model,feature_set,params,log_loss_mean,log_loss_std,accuracy_mean,accuracy_std
0,Bathrooms,With income,Logistic Regression,Extended with income,{'C': 0.1},0.751482,0.008724,0.657718,0.005400
1,Bathrooms,With income,Logistic Regression,Extended with income,{'C': 0.5},0.751510,0.008770,0.657441,0.005535
2,Bathrooms,With income,Logistic Regression,Extended with income,{'C': 5.0},0.751514,0.008724,0.657621,0.005540
3,Bathrooms,With income,Logistic Regression,Extended with income,{'C': 1.0},0.751582,0.008749,0.657920,0.005686
4,Bathrooms,With income,Logistic Regression,Extended with income,{'C': 10.0},0.751593,0.008725,0.657703,0.005776
5,Bathrooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",0.775916,0.008737,0.642320,0.006419
6,Bathrooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 200, 'learning_rate': 0.05, '...",0.776061,0.008769,0.641543,0.007656
7,Bathrooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 150, 'learning_rate': 0.1, 'm...",0.776452,0.008490,0.642163,0.007201
8,Bathrooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 150, 'learning_rate': 0.05, '...",0.777108,0.008434,0.641147,0.006538
9,Bathrooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 150, 'learning_rate': 0.05, '...",0.782200,0.007824,0.640537,0.006404


In [ ]:
# Select the minimum-log-loss hyperparameter configuration for each target and income scenario
best_tuning_results = tuning_results.groupby(["target", "income_scenario"], as_index=False).first()
best_tuning_results

,target,income_scenario,model,feature_set,params,log_loss_mean,log_loss_std,accuracy_mean,accuracy_std
0,Bathrooms,With income,Logistic Regression,Extended with income,{'C': 0.1},0.751482,0.008724,0.657718,0.005400
1,Bathrooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",0.775916,0.008737,0.642320,0.006419
2,Rooms,With income,Gradient Boosting,Extended with income,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",1.028903,0.005051,0.535585,0.005364
3,Rooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",1.040285,0.004649,0.531524,0.004753
4,Workers,With income,Random Forest,Extended with income,"{'n_estimators': 500, 'min_samples_leaf': 5, '...",1.018858,0.002501,0.531675,0.003450
5,Workers,Without income,Random Forest,Extended without income,"{'n_estimators': 500, 'min_samples_leaf': 10, ...",1.044977,0.003569,0.501113,0.003954


In [35]:
for _, result in best_tuning_results.iterrows():
    print(f"\n{'=' * 60}")
    print(f"{result['target']} - {result['income_scenario']}")
    print("=" * 60)
    print(f"Model: {result['model']}")
    print(f"Feature set: {result['feature_set']}")
    print(f"Best parameters: {result['params']}")
    print(f"CV weighted log loss: {result['log_loss_mean']:.4f} ± {result['log_loss_std']:.4f}")
    print(f"CV weighted accuracy: {result['accuracy_mean']:.4f} ± {result['accuracy_std']:.4f}")


Bathrooms - With income
Model: Logistic Regression
Feature set: Extended with income
Best parameters: {'C': 0.1}
CV weighted log loss: 0.7515 ± 0.0087
CV weighted accuracy: 0.6577 ± 0.0054

Bathrooms - Without income
Model: Gradient Boosting
Feature set: Extended without income
Best parameters: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 2}
CV weighted log loss: 0.7759 ± 0.0087
CV weighted accuracy: 0.6423 ± 0.0064

Rooms - With income
Model: Gradient Boosting
Feature set: Extended with income
Best parameters: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 2}
CV weighted log loss: 1.0289 ± 0.0051
CV weighted accuracy: 0.5356 ± 0.0054

Rooms - Without income
Model: Gradient Boosting
Feature set: Extended without income
Best parameters: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 2}
CV weighted log loss: 1.0403 ± 0.0046
CV weighted accuracy: 0.5315 ± 0.0048

Workers - With income
Model: Random Forest
Feature set: Extended with income
Best parameters: 

The tuning stage produces the following final configurations:

| Target | Income scenario | Selected model | Main hyperparameters |
| --- | --- | --- | --- |
| Bathrooms | With income | Logistic Regression | $C=0.1$ |
| Bathrooms | Without income | Gradient Boosting | 200 estimators, learning rate 0.1, depth 2 |
| Rooms | With income | Gradient Boosting | 200 estimators, learning rate 0.1, depth 2 |
| Rooms | Without income | Gradient Boosting | 200 estimators, learning rate 0.1, depth 2 |
| Workers | With income | Random Forest | 500 trees, minimum leaf size 5, maximum depth 20 |
| Workers | Without income | Random Forest | 500 trees, minimum leaf size 10, unrestricted maximum depth |

These configurations are selected entirely from cross-validation performance within the training partition. The reserved test samples remain unused up to this point.

### 3.4 Final test evaluation

The selected configuration for each target and income scenario is now fitted on its complete training partition and evaluated once on the corresponding reserved test sample.

This step estimates out-of-sample performance after all model-family, feature-set, and hyperparameter decisions have already been made.

The same survey-weighted log loss and accuracy metrics used during model selection are retained for consistency.

In [ ]:
test_results = []
test_pipelines = {}
# Evaluate each fully selected configuration once on its reserved test partition
for _, result in best_tuning_results.iterrows():
    target_name = result["target"]
    income_scenario = result["income_scenario"]
    model_name = result["model"]
    feature_set_name = result["feature_set"]
    params = result["params"]
    features = model_feature_sets[target_name][feature_set_name]
    target_column = target_columns[target_name]

    train = train_data[target_name]
    test = test_data[target_name]
    model = make_model(model_name, params)
    pipeline = build_pipeline(model_name, model, features)
    # Fit on the complete training partition before evaluating the held-out test data
    pipeline.fit(train[features], train[target_column], model__sample_weight=train[weight_column])

    test_probabilities = pipeline.predict_proba(test[features])
    test_predictions = pipeline.predict(test[features])
    classes = pipeline.named_steps["model"].classes_
    test_log_loss = log_loss(test[target_column], test_probabilities, labels=classes, sample_weight=test[weight_column])
    test_accuracy = accuracy_score(test[target_column], test_predictions, sample_weight=test[weight_column])

    # Retain the fitted test-stage pipeline for diagnostic purposes
    test_results.append({
        "target": target_name,
        "income_scenario": income_scenario,
        "model": model_name,
        "feature_set": feature_set_name,
        "params": params,
        "test_log_loss": test_log_loss,
        "test_accuracy": test_accuracy
    })

    if target_name not in test_pipelines:
        test_pipelines[target_name] = {}

    test_pipelines[target_name][income_scenario] = pipeline

    print(f"\n{'=' * 60}")
    print(f"{target_name} - {income_scenario}")
    print("=" * 60)
    print(f"Model: {model_name}")
    print(f"Feature set: {feature_set_name}")
    print(f"Test weighted log loss: {test_log_loss:.4f}")
    print(f"Test weighted accuracy: {test_accuracy:.4f}")


Bathrooms - With income
Model: Logistic Regression
Feature set: Extended with income
Test weighted log loss: 0.7539
Test weighted accuracy: 0.6580

Bathrooms - Without income
Model: Gradient Boosting
Feature set: Extended without income
Test weighted log loss: 0.7773
Test weighted accuracy: 0.6425

Rooms - With income
Model: Gradient Boosting
Feature set: Extended with income
Test weighted log loss: 1.0291
Test weighted accuracy: 0.5344

Rooms - Without income
Model: Gradient Boosting
Feature set: Extended without income
Test weighted log loss: 1.0380
Test weighted accuracy: 0.5304

Workers - With income
Model: Random Forest
Feature set: Extended with income
Test weighted log loss: 1.0264
Test weighted accuracy: 0.5247

Workers - Without income
Model: Random Forest
Feature set: Extended without income
Test weighted log loss: 1.0578
Test weighted accuracy: 0.4970


In [37]:
test_results = pd.DataFrame(test_results)
test_results

,target,income_scenario,model,feature_set,params,test_log_loss,test_accuracy
0,Bathrooms,With income,Logistic Regression,Extended with income,{'C': 0.1},0.753947,0.658041
1,Bathrooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",0.777262,0.642467
2,Rooms,With income,Gradient Boosting,Extended with income,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",1.029052,0.534383
3,Rooms,Without income,Gradient Boosting,Extended without income,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",1.038003,0.530381
4,Workers,With income,Random Forest,Extended with income,"{'n_estimators': 500, 'min_samples_leaf': 5, '...",1.026449,0.524683
5,Workers,Without income,Random Forest,Extended without income,"{'n_estimators': 500, 'min_samples_leaf': 10, ...",1.057795,0.496996


The test metrics are close to the corresponding cross-validation results for all six configurations.

| Target | Scenario | CV log loss | Test log loss | Test accuracy |
| --- | --- | ---: | ---: | ---: |
| Bathrooms | With income | 0.7515 | 0.7539 | 0.6580 |
| Bathrooms | Without income | 0.7759 | 0.7773 | 0.6425 |
| Rooms | With income | 1.0289 | 1.0291 | 0.5344 |
| Rooms | Without income | 1.0403 | 1.0380 | 0.5304 |
| Workers | With income | 1.0189 | 1.0264 | 0.5247 |
| Workers | Without income | 1.0450 | 1.0578 | 0.4970 |

No substantial deterioration appears when moving from cross-validation to the held-out test samples.

The objective is not to assign each household deterministically to its most likely class. The probability vectors produced by these models are retained and propagated into the AMAI score calculation, so log loss remains the main metric for assessing the predictions.

## 4. Final model fitting and application to the OD survey

### 4.1 Refit using all available ENIGH observations

The test evaluation completes the model-selection process. The selected model architecture, feature set, and hyperparameters are therefore fixed.

Before transferring the models to the OD survey, each selected pipeline is fitted again using all ENIGH observations with an observed value for the corresponding target:

$$\mathcal{D}_{\mathrm{final}}^{(t)} = \mathcal{D}_{\mathrm{train}}^{(t)} \cup \mathcal{D}_{\mathrm{test}}^{(t)}.$$

This final refit does not constitute an additional model-selection step. It allows the deployed model to use all labeled ENIGH information after its configuration has already been evaluated.Considering the complete data from the ENIGH database, we now retrain the model before applying it to the OD survey.

In [ ]:
final_pipelines = {}
# Refit the selected configurations using all labeled ENIGH observations
for _, result in best_tuning_results.iterrows():
    target_name = result["target"]
    income_scenario = result["income_scenario"]
    model_name = result["model"]
    feature_set_name = result["feature_set"]
    params = result["params"]

    target_column = target_columns[target_name]
    features = model_feature_sets[target_name][feature_set_name]
    data = model_data[target_name]

    model = make_model(model_name, params)
    pipeline = build_pipeline(model_name, model, features)

    pipeline.fit(data[features], data[target_column], model__sample_weight=data[weight_column])

    if target_name not in final_pipelines:
        final_pipelines[target_name] = {}

    # Store the complete fitted pipeline together with the information required for OD prediction
    final_pipelines[target_name][income_scenario] = {
        "pipeline": pipeline,
        "features": features,
        "classes": pipeline.named_steps["model"].classes_,
        "model": model_name,
        "parameters": params,
        "feature_set": feature_set_name
    }

    print(f"{target_name} - {income_scenario}: trained on {len(data):,} observations")

Bathrooms - With income: trained on 88,364 observations
Bathrooms - Without income: trained on 88,364 observations
Rooms - With income: trained on 90,102 observations
Rooms - Without income: trained on 90,102 observations
Workers - With income: trained on 90,102 observations
Workers - Without income: trained on 90,102 observations


### 4.2 Selection of the income scenario in OD

Each OD dwelling is assigned to one of the two prediction scenarios according to the availability of `ingreso_hogar`.

The OD data contain 17,901 dwellings:

- 7,469 with observed household income;
- 10,432 without household income.

The first group receives predictions from the corresponding with-income model, while the second group receives predictions from the corresponding without-income model.

In [ ]:
probability_column_mappings = {
    "Workers": {
        "0": "prob_workers_0",
        "1": "prob_workers_1",
        "2": "prob_workers_2",
        "3": "prob_workers_3",
        "4 o más": "prob_workers_4_mas"
    },
    "Bathrooms": {
        "0": "prob_bathrooms_0",
        "1": "prob_bathrooms_1",
        "2 o más": "prob_bathrooms_2_mas"
    },
    "Rooms": {
        "1": "prob_rooms_1",
        "2": "prob_rooms_2",
        "3": "prob_rooms_3",
        "4 o más": "prob_rooms_4_mas"
    }
}

# Assign OD dwellings to the with-income or without-income prediction scenario
income_known_mask = od_housing["ingreso_hogar"].notna()
income_missing_mask = od_housing["ingreso_hogar"].isna()

print(f"OD observations with income: {income_known_mask.sum():,}")
print(f"OD observations without income: {income_missing_mask.sum():,}")
print(f"Total OD observations: {len(od_housing):,}")

OD observations with income: 7,469
OD observations without income: 10,432
Total OD observations: 17,901


### 4.3 Hybrid probabilistic prediction

The six fitted pipelines are now combined into three hybrid models, one for each AMAI component.

For target $t$, each OD dwelling receives a probability vector

$$\widehat{\mathbf p}_h^{(t)} = \left(\hat p_{h1}^{(t)}, \ldots, \hat p_{hK_t}^{(t)} \right),$$

using either the with-income or without-income pipeline according to the rule defined previously.

For every target and dwelling,

$$\sum_{k=1}^{K_t} \hat p_{hk}^{(t)} = 1.$$

The individual class probabilities are retained as columns in the OD dataset rather than collapsing them to a single predicted category.

In [ ]:
for target_name, probability_mapping in probability_column_mappings.items():
    probability_columns = list(probability_mapping.values())

    # Initialize the probability columns for the current AMAI target
    for probability_column in probability_columns:
        od_housing[probability_column] = np.nan

    # Use the with-income pipeline for dwellings with observed household income
    model_info = final_pipelines[target_name]["With income"]
    pipeline = model_info["pipeline"]
    features = model_info["features"]
    classes = list(model_info["classes"])

    probabilities = pipeline.predict_proba(od_housing.loc[income_known_mask, features])

    for class_name, probability_column in probability_mapping.items():
        class_index = classes.index(class_name)
        od_housing.loc[income_known_mask, probability_column] = probabilities[:, class_index]

    # Use the without-income pipeline for dwellings with missing household income
    model_info = final_pipelines[target_name]["Without income"]
    pipeline = model_info["pipeline"]
    features = model_info["features"]
    classes = list(model_info["classes"])

    probabilities = pipeline.predict_proba(od_housing.loc[income_missing_mask, features])

    for class_name, probability_column in probability_mapping.items():
        class_index = classes.index(class_name)
        od_housing.loc[income_missing_mask, probability_column] = probabilities[:, class_index]

As a consistency check, we verify that the probabilities generated for each target sum to one for every OD dwelling.

In [ ]:
# Sanity check: class probabilities must sum to one for every dwelling and target
for target_name, probability_mapping in probability_column_mappings.items():
    probability_columns = list(probability_mapping.values())
    probability_sum_column = f"{target_name.lower()}_probability_sum"

    od_housing[probability_sum_column] = od_housing[probability_columns].sum(axis=1)

    print(f"\n{target_name}")
    print(od_housing[probability_sum_column].describe())


Workers
count    1.790100e+04
mean     1.000000e+00
std      4.481756e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: workers_probability_sum, dtype: float64

Bathrooms
count    1.790100e+04
mean     1.000000e+00
std      7.981813e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: bathrooms_probability_sum, dtype: float64

Rooms
count    1.790100e+04
mean     1.000000e+00
std      1.005144e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: rooms_probability_sum, dtype: float64


In [42]:
od_housing

,folio_vivienda,fecha,municipio,ageb,centralidad,problema_movilidad_principal,problema_tp_principal,personas_en_vivienda,tenencia_vivienda,ingreso_mensual_hogar,...,prob_bathrooms_0,prob_bathrooms_1,prob_bathrooms_2_mas,prob_rooms_1,prob_rooms_2,prob_rooms_3,prob_rooms_4_mas,workers_probability_sum,bathrooms_probability_sum,rooms_probability_sum
0,1,2023-01-25 00:00:00+00:00,Tlajomulco,1409700251418,30F,Tiempos excesivos de traslado,La distancia a las paradas,5,Propia,"Más de $25,001",...,0.001520,0.276486,0.721995,0.010185,0.116451,0.470905,0.402459,1.0,1.0,1.0
1,2,2023-01-26 00:00:00+00:00,Guadalajara,1403900010859,09,Congestionamiento,El sobrecupo de personas en las unidades del t...,3,Rentada,"$3,001 y $7,000",...,0.229989,0.704064,0.065947,0.204763,0.620898,0.153914,0.020425,1.0,1.0,1.0
2,3,2023-01-26 00:00:00+00:00,Guadalajara,1403900010539,09,Pavimento deteriorado,La distancia a las paradas,3,Prestada,No quiso responder,...,0.305223,0.623315,0.071462,0.593559,0.306492,0.083356,0.016593,1.0,1.0,1.0
3,4,2023-01-26 00:00:00+00:00,Guadalajara,1403900010539,09,Falta de señalamientos,El tiempo de duración de viaje,3,Rentada,"Más de $25,001",...,0.014172,0.658521,0.327307,0.061466,0.505851,0.385099,0.047584,1.0,1.0,1.0
4,5,2023-01-26 00:00:00+00:00,Guadalajara,1403900010539,09,Congestionamiento,El sobrecupo de personas en las unidades del t...,6,Propia,"$7,001 y $10,000",...,0.319899,0.581980,0.098121,0.067207,0.428722,0.360466,0.143605,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17896,17963,2023-02-17 00:00:00+00:00,Tlaquepaque,1409800013322,18,Falta de estacionamiento,Trato del/de la chofer,4,Rentada,"$7,001 y $10,000",...,0.200971,0.733940,0.065089,0.157191,0.615691,0.200965,0.026154,1.0,1.0,1.0
17897,17964,2023-02-17 00:00:00+00:00,Tlaquepaque,140980001110A,18,Falta de cruces peatonales,Trato del/de la chofer,6,Rentada,"$15,001 y $25,000",...,0.176752,0.761844,0.061404,0.179103,0.568744,0.200979,0.051174,1.0,1.0,1.0
17898,17965,2023-02-17 00:00:00+00:00,Tlaquepaque,140980001110A,18,Insuficiente transporte público,El costo del viaje,2,Rentada,"$10,001 y $15,000",...,0.059937,0.825103,0.114960,0.403106,0.496601,0.085635,0.014658,1.0,1.0,1.0
17899,17966,2023-04-15 00:00:00+00:00,Tonalá,1410100261709,45,Insuficiente transporte público,Falta de cubiertas para espera en las paradas,5,Rentada,No quiso responder,...,0.076233,0.803806,0.119961,0.103113,0.586649,0.271636,0.038602,1.0,1.0,1.0


### 4.4 Probability-weighted AMAI scores

The final AMAI calculation requires scores rather than discrete predicted categories.

Let $s_k^{(t)}$ denote the AMAI score associated with category $k$ of component $t$. Instead of assigning each household to

$$\arg\max_k \hat p_{hk}^{(t)},$$

we retain the uncertainty of the prediction and calculate the expected AMAI score:

$$ \widehat S_h^{(t)} = \mathbb{E} \left[S_h^{(t)} \mid X_h \right] = \sum_{k=1}^{K_t} \hat p_{hk}^{(t)} s_k^{(t)}$$

This produces one continuous expected score for each of the three modeled components:

$$\widehat S_h^{(\mathrm{workers})}, \qquad \widehat S_h^{(\mathrm{bathrooms})}, \qquad \widehat S_h^{(\mathrm{rooms})}.$$

The procedure is consistent with the probabilistic treatment of head-of-household education introduced in the previous notebook: uncertainty is retained until the component is converted to its AMAI score.

In [ ]:
# AMAI score assigned to each possible category of the modeled components
workers_score_mapping = {
    "0": 0,
    "1": 15,
    "2": 31,
    "3": 46,
    "4 o más": 61
}

bathrooms_score_mapping = {
    "0": 0,
    "1": 24,
    "2 o más": 47
}

rooms_score_mapping = {
    "1": 8,
    "2": 16,
    "3": 24,
    "4 o más": 32
}

# Calculate probability-weighted expected AMAI scores for each OD dwelling
od_housing["modelo_puntos_14_anos_trabajo_mes"] = sum(od_housing[probability_column_mappings["Workers"][class_name]] * workers_score_mapping[class_name] for class_name in workers_score_mapping)
od_housing["modelo_puntos_banos_completos"] = sum(od_housing[probability_column_mappings["Bathrooms"][class_name]] * bathrooms_score_mapping[class_name] for class_name in bathrooms_score_mapping)
od_housing["modelo_puntos_cuartos_dormir"] = sum(od_housing[probability_column_mappings["Rooms"][class_name]] * rooms_score_mapping[class_name] for class_name in rooms_score_mapping)

In [ ]:
# Remove temporary columns used only to validate the probability vectors
od_housing.drop(columns=["workers_probability_sum", "bathrooms_probability_sum", "rooms_probability_sum"], inplace=True)

In [46]:
od_housing

,folio_vivienda,fecha,municipio,ageb,centralidad,problema_movilidad_principal,problema_tp_principal,personas_en_vivienda,tenencia_vivienda,ingreso_mensual_hogar,...,prob_bathrooms_0,prob_bathrooms_1,prob_bathrooms_2_mas,prob_rooms_1,prob_rooms_2,prob_rooms_3,prob_rooms_4_mas,modelo_puntos_14_anos_trabajo_mes,modelo_puntos_banos_completos,modelo_puntos_cuartos_dormir
0,1,2023-01-25 00:00:00+00:00,Tlajomulco,1409700251418,30F,Tiempos excesivos de traslado,La distancia a las paradas,5,Propia,"Más de $25,001",...,0.001520,0.276486,0.721995,0.010185,0.116451,0.470905,0.402459,38.910128,40.569402,26.125103
1,2,2023-01-26 00:00:00+00:00,Guadalajara,1403900010859,09,Congestionamiento,El sobrecupo de personas en las unidades del t...,3,Rentada,"$3,001 y $7,000",...,0.229989,0.704064,0.065947,0.204763,0.620898,0.153914,0.020425,22.869987,19.997027,15.920013
2,3,2023-01-26 00:00:00+00:00,Guadalajara,1403900010539,09,Pavimento deteriorado,La distancia a las paradas,3,Prestada,No quiso responder,...,0.305223,0.623315,0.071462,0.593559,0.306492,0.083356,0.016593,21.838894,18.318272,12.183860
3,4,2023-01-26 00:00:00+00:00,Guadalajara,1403900010539,09,Falta de señalamientos,El tiempo de duración de viaje,3,Rentada,"Más de $25,001",...,0.014172,0.658521,0.327307,0.061466,0.505851,0.385099,0.047584,30.569045,31.187937,19.350402
4,5,2023-01-26 00:00:00+00:00,Guadalajara,1403900010539,09,Congestionamiento,El sobrecupo de personas en las unidades del t...,6,Propia,"$7,001 y $10,000",...,0.319899,0.581980,0.098121,0.067207,0.428722,0.360466,0.143605,33.004357,18.579225,20.643750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17896,17963,2023-02-17 00:00:00+00:00,Tlaquepaque,1409800013322,18,Falta de estacionamiento,Trato del/de la chofer,4,Rentada,"$7,001 y $10,000",...,0.200971,0.733940,0.065089,0.157191,0.615691,0.200965,0.026154,27.562125,20.673759,16.768643
17897,17964,2023-02-17 00:00:00+00:00,Tlaquepaque,140980001110A,18,Falta de cruces peatonales,Trato del/de la chofer,6,Rentada,"$15,001 y $25,000",...,0.176752,0.761844,0.061404,0.179103,0.568744,0.200979,0.051174,29.464439,21.170237,16.993787
17898,17965,2023-02-17 00:00:00+00:00,Tlaquepaque,140980001110A,18,Insuficiente transporte público,El costo del viaje,2,Rentada,"$10,001 y $15,000",...,0.059937,0.825103,0.114960,0.403106,0.496601,0.085635,0.014658,16.672766,25.205581,13.694759
17899,17966,2023-04-15 00:00:00+00:00,Tonalá,1410100261709,45,Insuficiente transporte público,Falta de cubiertas para espera en las paradas,5,Rentada,No quiso responder,...,0.076233,0.803806,0.119961,0.103113,0.586649,0.271636,0.038602,32.528530,24.929520,17.965813


## 5. Final outputs

The resulting OD dwelling dataset now contains:

- the AMAI-related variables inherited from the previous notebook;
- class probabilities for the number of workers aged 14 or older;
- class probabilities for the number of complete bathrooms;
- class probabilities for the number of sleeping rooms; and
- the probability-weighted AMAI score associated with each of these three modeled components.

The final dwelling-level dataset is saved as `od_housing_amai_imputed.csv`.

The fitted models are also serialized in `amai_imputation_models.joblib`. The saved bundle contains the final pipelines, target definitions, feature sets, selected hyperparameters, probability-column mappings, survey-weight variable, and AMAI score mappings required to reproduce the imputation process.

This notebook therefore completes the estimation of the AMAI household components that are not directly observed in the OD survey. Their aggregation into the final socioeconomic score and socioeconomic-level category can be performed in the subsequent stage.

In [47]:
from pathlib import Path
import joblib

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

files_output_directory = ROOT / "outputs" / "files"
models_output_directory = ROOT / "outputs" / "models"

files_output_directory.mkdir(parents=True, exist_ok=True)
models_output_directory.mkdir(parents=True, exist_ok=True)

In [ ]:
# Save the OD dwelling dataset with probabilistic AMAI component estimates
od_housing.to_csv(files_output_directory / "od_housing_amai_imputed.csv", index=False, encoding="utf-8-sig")

# Bundle the fitted pipelines and metadata required to reproduce the imputation
amai_model_bundle = {
    "pipelines": final_pipelines,
    "targets": target_columns,
    "feature_sets": model_feature_sets,
    "best_tuning_results": best_tuning_results,
    "weight_column": weight_column,
    "probability_column_mappings": probability_column_mappings,
    "score_mappings": {
        "Workers": workers_score_mapping,
        "Bathrooms": bathrooms_score_mapping,
        "Rooms": rooms_score_mapping
    }
}

# Serialize all AMAI imputation models in a single reusable file
joblib.dump(amai_model_bundle, models_output_directory / "amai_imputation_models.joblib")

['/Users/sebastiangutierrezbernal/Desktop/Tec/Ciudades para el futuro/Proyecto Transporte GDL/informal_jobs/outputs/models/amai_imputation_models.joblib']